# CLT trên giá trị đơn hàng Olist: "n ≥ 30 là đủ" có đúng không?

Giá trị đơn (tổng `price + freight_value` của mọi item trong đơn) lệch phải rất mạnh: vài đơn hơn
10 nghìn BRL bên cạnh hàng chục nghìn đơn dưới 100 BRL. CLT nói trung bình mẫu sẽ xấp xỉ chuẩn khi
n đủ lớn. Bài này đo xem "đủ lớn" là bao nhiêu với dữ liệu thật, bằng cách coi toàn bộ đơn giao xong
là **tổng thể** và lấy mẫu từ đó.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm.
- Cell gắn tag `explore` (dòng đầu `# explore`) là chỗ đọc dữ liệu thật, vẽ, gọi hàm. Test bỏ qua chúng.
- Chấm: `uv run pytest tracks/data-science/modules/02-probability-statistics/exercises/intermediate/01-clt-order-value -q`

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

In [ ]:
# explore: đọc Olist thật (tải bằng: uv run python data/download_olist.py)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data/raw"
orders = pd.read_csv(RAW / "olist_orders_dataset.csv", parse_dates=["order_delivered_customer_date"])
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")

delivered = orders[(orders.order_status == "delivered") & orders.order_delivered_customer_date.notna()]
value = (items.assign(v=items.price + items.freight_value)
              .groupby("order_id").v.sum()
              .reindex(delivered.order_id).dropna().to_numpy())
print(len(value), "đơn giao xong")
pd.Series(value).describe(percentiles=[0.5, 0.9, 0.99])

In [ ]:
# explore: lệch phải cỡ nào? So mean với median, skewness, và phần doanh thu của 1% đơn lớn nhất
top = np.sort(value)[::-1][: len(value) // 100]
print(f"mean {value.mean():.2f}  median {np.median(value):.2f}  skew {stats.skew(value):.2f}")
print(f"1% đơn lớn nhất chiếm {top.sum() / value.sum():.1%} doanh thu")

In [ ]:
# explore: Law of Large Numbers: trung bình chạy (running mean) của một chuỗi đơn ngẫu nhiên
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
stream = rng.choice(value, 20000)
running = np.cumsum(stream) / np.arange(1, len(stream) + 1)
plt.plot(running)
plt.axhline(value.mean(), color="k", ls="--")
plt.xscale("log")
plt.xlabel("số đơn đã thấy")
plt.ylabel("trung bình chạy (BRL)")
plt.show()

## 1. Phân phối của trung bình mẫu

Viết `sample_means(x, n, n_sims=2000, seed=0)`: lấy `n_sims` mẫu, mỗi mẫu `n` phần tử **có hoàn
lại** từ `x`, trả về mảng `n_sims` trung bình mẫu. Dùng `np.random.default_rng(seed)`. Vector hoá
(một ma trận chỉ số `n_sims × n`), đừng lặp Python.

In [ ]:
def sample_means(x: np.ndarray, n: int, n_sims: int = 2000, seed: int = 0) -> np.ndarray:
    """n_sims trung bình mẫu, mỗi mẫu n phần tử lấy có hoàn lại từ x."""
    x = np.asarray(x, dtype=float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(x), size=(n_sims, n))
    return x[idx].mean(axis=1)

In [ ]:
# explore: histogram trung bình mẫu với n = 5, 30, 500. Skewness giảm cỡ skew(x) / sqrt(n)
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, n in zip(axes, [5, 30, 500]):
    m = sample_means(value, n, 5000)
    ax.hist(m, bins=80)
    ax.set_title(f"n = {n}, skew = {stats.skew(m):.2f} (lý thuyết {stats.skew(value) / np.sqrt(n):.2f})")
plt.show()

## 2. Độ phủ (coverage) của khoảng tin cậy t

Khoảng t 95% `mean ± t_{0.975, n-1} · s / sqrt(n)` chỉ đúng 95% khi trung bình mẫu xấp xỉ chuẩn.
Viết `t_interval_coverage(x, n, n_sims=2000, seed=0, alpha=0.05)`: rút `n_sims` mẫu cỡ `n` từ
`x` (có hoàn lại, cùng cách như trên), dựng khoảng t (`s` với `ddof=1`) cho mỗi mẫu, trả tỉ lệ
khoảng chứa `x.mean()` (trung bình **tổng thể**).

In [ ]:
def t_interval_coverage(x: np.ndarray, n: int, n_sims: int = 2000, seed: int = 0,
                        alpha: float = 0.05) -> float:
    """Tỉ lệ khoảng t (1 - alpha) dựng trên mẫu cỡ n chứa trung bình tổng thể x.mean()."""
    x = np.asarray(x, dtype=float)
    rng = np.random.default_rng(seed)
    samples = x[rng.integers(0, len(x), size=(n_sims, n))]
    means = samples.mean(axis=1)
    half = stats.t.ppf(1 - alpha / 2, n - 1) * samples.std(axis=1, ddof=1) / np.sqrt(n)
    return float(np.mean(np.abs(means - x.mean()) <= half))

In [ ]:
# explore: bảng coverage theo n trên giá trị đơn thật
pd.Series({n: t_interval_coverage(value, n, 4000) for n in [5, 30, 100, 500, 2000]}, name="coverage")

In [ ]:
# explore: vì sao khoảng hụt? Ở mẫu nhỏ, mean và s tương quan dương: mẫu "trúng" đơn lớn có
# mean cao VÀ s lớn (khoảng rộng), mẫu không có đơn lớn có mean thấp VÀ khoảng hẹp → trượt xuống dưới.
m = sample_means(value, 30, 4000, seed=1)
print("tỉ lệ trung bình mẫu < trung bình tổng thể:", np.mean(m < value.mean()))

## 3. Cần bao nhiêu mẫu?

Viết `min_n_for_coverage(x, ns, target=0.93, n_sims=2000, seed=0)`: duyệt `ns` theo thứ tự tăng
dần, trả về `n` **nhỏ nhất** có `t_interval_coverage(x, n, n_sims, seed) >= target`; không có thì
trả `None`.

In [ ]:
def min_n_for_coverage(x: np.ndarray, ns: list[int], target: float = 0.93, n_sims: int = 2000,
                       seed: int = 0) -> int | None:
    """n nhỏ nhất trong ns đạt coverage >= target, hoặc None."""
    for n in sorted(ns):
        if t_interval_coverage(x, n, n_sims, seed) >= target:
            return int(n)
    return None

In [ ]:
# explore: so giá trị đơn với log giá trị đơn (gần chuẩn hơn nhiều)
ns = [5, 10, 30, 100, 300, 1000, 3000]
print("giá trị đơn:", min_n_for_coverage(value, ns))
print("log giá trị đơn:", min_n_for_coverage(np.log(value), ns))

## 4. Kết luận cho team

Coi 96 470 đơn giao xong là tổng thể: giá trị đơn có mean 159.83, median 105.28, skewness 9.37, và
1% đơn lớn nhất chiếm 10.4% doanh thu. Với n = 30, khoảng t "95%" chỉ phủ khoảng 84% số lần; n = 500
được khoảng 93%, và phải tới n = 1000 mới qua ngưỡng 93% trong lưới đã thử. Nguyên nhân: skewness
của trung bình mẫu chỉ giảm theo 9.37 / sqrt(n), và mẫu nhỏ thường "hụt" đơn lớn nên khoảng trượt
xuống dưới (57% trung bình mẫu n = 30 nằm dưới trung bình tổng thể). Trên log giá trị đơn, n = 5
đã đủ. Với mẫu nhỏ: báo cáo median kèm bootstrap CI, hoặc phân tích trên log và nói rõ đang nói về
trung bình hình học. A/B test trong module 03 có hàng nghìn đơn mỗi nhánh nên t-test cho chênh lệch
trung bình vẫn dùng được; khi có "cá voi" thì kiểm tra lại bằng bootstrap.